# SAT solvers — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def truth_rows(names):
    return [dict(zip(names, vals)) for vals in itertools.product([False, True], repeat=len(names))]
def draw_graph(nodes, edges, title='graph'):
    angles=np.linspace(0, 2*np.pi, len(nodes), endpoint=False)
    pos={n:(np.cos(a), np.sin(a)) for n,a in zip(nodes, angles)}
    plt.figure(figsize=(5,4))
    for a,b in edges:
        xa,ya=pos[a]; xb,yb=pos[b]; plt.plot([xa,xb],[ya,yb], 'k-', alpha=.6)
    for n,(x,y) in pos.items():
        plt.scatter([x],[y], s=500, zorder=3); plt.text(x,y,n,ha='center',va='center',color='white',weight='bold')
    plt.axis('equal'); plt.axis('off'); plt.title(title)
    return pos
print('setup ok')

## Searching Boolean assignments for a satisfying one
SAT solving asks whether a CNF formula has a truth assignment that makes every clause true. The examples visualize clauses, brute force, DPLL unit propagation, pure literals, and a tiny search tree.

In [ ]:
# 1) evaluate a CNF: (A or B) and (not A or C)
rows=truth_rows(['A','B','C']); sat=[(r['A'] or r['B']) and ((not r['A']) or r['C']) for r in rows]
plt.figure(figsize=(7,3)); plt.bar(range(8), sat, color=['tab:green' if s else 'tab:red' for s in sat]); plt.xticks(range(8), [f"{int(r['A'])}{int(r['B'])}{int(r['C'])}" for r in rows]); plt.title('satisfying assignments')
assert sum(sat)==4

In [ ]:
# 2) brute force checks every assignment
checks=list(range(1,9)); found=next(i for i,s in enumerate(sat,1) if s)
plt.figure(figsize=(5,3)); plt.plot(checks, [i>=found for i in checks], '-o'); plt.title('first satisfying assignment found')
assert found==3

In [ ]:
# 3) unit propagation: clause (A) forces A=True
clauses=[[('A',True)],[('A',False),('C',True)]]; assign={'A':True}; remaining=[[lit for lit in c if lit[0] not in assign] for c in clauses if not any(assign.get(v)==pos for v,pos in c)]
plt.figure(figsize=(5,3)); plt.bar(['clauses before','after unit'], [2,len(remaining)], color='tab:orange'); plt.title('unit propagation simplifies CNF')
assert remaining==[[('C', True)]]

In [ ]:
# 4) pure literal: B appears only positively, so set B=True
clauses=[[('A',True),('B',True)],[('A',False),('B',True)]]; signs={}
for c in clauses:
    for v,pos in c: signs.setdefault(v,set()).add(pos)
pure=[v for v,s in signs.items() if len(s)==1]
plt.figure(figsize=(5,3)); plt.bar(['A','B'], [len(signs['A']),len(signs['B'])], color=['tab:red','tab:green']); plt.title('B is pure')
assert pure==['B']

In [ ]:
# 5) search tree nodes shrink after propagation
nodes=['root','A=True','C=True']; edges=[('root','A=True'),('A=True','C=True')]; draw_graph(nodes,edges,'DPLL forced path')
assert len(nodes)==3